In [ ]:
!pip install -q beautifulsoup4 requests

In [ ]:
import os
import re
import json
import time
import requests

from collections import deque
from bs4 import BeautifulSoup
from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive


drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (compatible; UPV-KB-Bot/1.0)"
}

PAUSA = 0.3

PROFUNDIDAD_MAX = 1


FUENTES = [

    (
        "admision_grado.json",
        "https://www.upv.es/admision/admision-grado/bachillerato-es.html"
    ),

    (
        "admision_master.json",
        "https://www.upv.es/admision/admision-master/index-es.html"
    ),

    (
        "admision_doctorado.json",
        "https://www.upv.es/admision/admision-doctorado/index-es.html"
    ),

    (
        "internacional.json",
        "https://www.upv.es/admision/internacional/"
    )

]

In [ ]:
NOMBRE_PROGRAMA = "JSONs_Admision.ipynb"     # cambiar si procede


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:
        ruta_programa = root
        break


if ruta_programa is None:
    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(
    ruta_programa,
    "JSONs"
)

os.makedirs(
    CARPETA_JSON,
    exist_ok=True
)

In [ ]:
def limpiar_texto(txt):
    return re.sub(r"\s+", " ", txt).strip()


def normalizar_url(url):
    p = urlparse(url)
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))


def get(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        time.sleep(PAUSA)
        return BeautifulSoup(r.text, "html.parser")
    except Exception:
        return None

In [ ]:
for nombre_json, url in FUENTES:

    print()
    print("="*70)
    print(nombre_json)
    print("="*70)

    paginas = recorrer_fuente(url)


    datos = {
        "fuente": url,
        "total": len(paginas),
        "paginas": paginas
    }


    ruta = os.path.join(
        CARPETA_JSON,
        nombre_json
    )


    with open(
        ruta,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            datos,
            f,
            ensure_ascii=False,
            indent=2
        )


    print()
    print(
        f"Guardadas {len(paginas)} páginas."
    )

    print(ruta)


print()
print("Proceso terminado.")


admision_grado.json
[0] https://www.upv.es/admision/admision-grado/bachillerato-es.html
[1] https://www.upv.es/
[1] https://www.upv.es/admision/admision-master/index-es.html
[1] https://www.upv.es/admision/admision-doctorado/index-es.html
[1] https://www.upv.es/admision/internacional/
[1] https://www.upv.es/estudios/grado/index-es.html
[1] https://www.upv.es/estudios/master/index-es.html
[1] https://www.upv.es/entidades/edoctorado/oferta-de-programas-de-doctorado/
[1] https://www.upv.es/investigacion/estructuras/index-es.html
[1] https://www.upv.es/investigacion/iniciativas-idi/index-es.html
[1] https://www.upv.es/organizacion/la-institucion/index-es.html
[1] https://www.upv.es/organizacion/escuelas-facultades/index-es.html
[1] https://www.upv.es/organizacion/departamentos/index-es.html
[1] https://www.upv.es/organizacion/servicios-universitarios/index-es.html
[1] https://www.upv.es/contenidos/global/
[1] https://www.upv.es/organizacion/sostenibilidad/index-es.html
[1] https://www.upv

In [ ]:
url = "https://www.upv.es/admision/admision-grado/bachillerato-es.html"

soup = get(url)

candidatos = []

for tag in soup.find_all(["main", "article", "section", "div"]):

    enlaces = len(tag.find_all("a", href=True))

    if tag.get("id") or tag.get("class"):

        candidatos.append({
            "enlaces": enlaces,
            "tag": tag.name,
            "id": tag.get("id"),
            "class": tag.get("class")
        })

candidatos = sorted(
    candidatos,
    key=lambda x: x["enlaces"],
    reverse=True
)

for c in candidatos[:20]:
    print(c)

{'enlaces': 130, 'tag': 'div', 'id': 'smooth-wrapper', 'class': None}
{'enlaces': 84, 'tag': 'main', 'id': None, 'class': ['search-page']}
{'enlaces': 27, 'tag': 'div', 'id': None, 'class': ['menu', 'bg-white']}
{'enlaces': 27, 'tag': 'div', 'id': None, 'class': ['grid', 'grid-menu', 'align-center']}
{'enlaces': 18, 'tag': 'div', 'id': None, 'class': ['container', 'footer-cols']}
{'enlaces': 18, 'tag': 'div', 'id': None, 'class': ['grid-4', 'grid']}
{'enlaces': 14, 'tag': 'section', 'id': 'section-03', 'class': ['container']}
{'enlaces': 14, 'tag': 'div', 'id': None, 'class': ['grid', 'grid-12']}
{'enlaces': 14, 'tag': 'div', 'id': None, 'class': ['col-8', 'has-details']}
{'enlaces': 12, 'tag': 'section', 'id': 'section-02', 'class': ['container']}
{'enlaces': 12, 'tag': 'div', 'id': None, 'class': ['grid', 'grid-12']}
{'enlaces': 12, 'tag': 'div', 'id': None, 'class': ['col-8']}
{'enlaces': 12, 'tag': 'section', 'id': 'section-04', 'class': ['container']}
{'enlaces': 12, 'tag': 'div',

In [ ]:
main = soup.find("main")

print(len(main.find_all("a", href=True)))

84


In [ ]:
for a in main.find_all("a", href=True):
    print(a.get_text(" ", strip=True), "->", a["href"])

Bachillerato -> ./bachillerato-es.html
Ciclos formativos -> ./ciclos-formativos-es.html
Titulados universitarios -> ./titulados-universitarios-es.html
Mayores de 25/40/45 años -> ./mayores-25-40-45-es.html
Vengo de otra universidad -> ./vengo-de-otra-universidad-es.html
 -> 
Ir a la sección -> #section-01
Ir a la sección -> #section-02
Ir a la sección -> #section-03
Ir a la sección -> #section-04
Ir a la sección -> #section-05
Ir a la sección -> #section-06
Más información -> https://www.jpa.upv.es/
Más información -> https://www.upv.es/contenidos/jpa/sesiones-on-line-2/#videopodcasts-open
Más información -> https://www.upv.es/contenidos/embajadores/
Más información -> https://www.upv.es/contenidos/praktikum/
Más información -> https://geocaching.upv.es/portada/cas/index.html
25 razones para decidirte -> https://www.upv.es/perfiles/futuro-alumno/veinte-razones-es.html
Generación Espontánea -> https://generacionespontanea.upv.es/
La UPV en los rankings -> https://www.upv.es/rankings/ind